# Food Delivery Marketplace Analytics
## 02 — Revenue RCA & Mathematical Decomposition

Detailed analysis of the ~10% GMV contraction from April 2018 (peak) to May 2018.
Isolates exact drivers using three-factor revenue decomposition.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text

BASE_DIR = Path().resolve().parent
sys.path.insert(0, str(BASE_DIR))

from src.db_connect import get_engine, execute_query

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Environment initialized")

## 1. Load and Prepare Data

Extract orders for comparison periods: April 2018 (baseline) vs May 2018 (contraction).

In [ ]:
engine, dialect = get_engine()

# Load fact tables
with engine.connect() as conn:
    df_orders = pd.read_sql("SELECT * FROM fact_orders", conn)
    df_order_items = pd.read_sql("SELECT * FROM fact_order_items", conn)
    df_customers = pd.read_sql("SELECT * FROM dim_customers", conn)
    df_products = pd.read_sql("SELECT * FROM dim_products", conn)

# Convert timestamps
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])

# Extract GMV per order
order_gmv = df_order_items.groupby('order_id').agg({
    'price': 'sum',
    'freight_value': 'sum'
}).reset_index()
order_gmv['order_gmv'] = order_gmv['price'] + order_gmv['freight_value']

# Merge
df_orders = df_orders.merge(order_gmv[['order_id', 'order_gmv']], on='order_id', how='left')
df_orders = df_orders.merge(df_customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')
df_orders['order_month'] = df_orders['order_purchase_timestamp'].dt.strftime('%Y-%m')

# Filter: Delivered orders only (for GMV accuracy)
delivered = df_orders[df_orders['order_status'] == 'delivered'].copy()

print(f"Orders loaded: {len(df_orders):,} total, {len(delivered):,} delivered")

## 2. Three-Factor Revenue Decomposition

Mathematical model: **GMV = Users × Order Frequency × AOV**

Changes in GMV can be attributed to changes in user volume, order frequency, or average order value.

In [ ]:
# Aggregate by month (delivered orders only)
monthly_data = delivered.groupby('order_month').agg({
    'customer_unique_id': 'nunique',  # Active users
    'order_id': 'count',              # Total orders
    'order_gmv': 'sum'                # Total GMV
}).reset_index()

monthly_data.columns = ['order_month', 'active_users', 'total_orders', 'total_gmv']
monthly_data['order_frequency'] = monthly_data['total_orders'] / monthly_data['active_users']
monthly_data['aov'] = monthly_data['total_gmv'] / monthly_data['total_orders']

print("Monthly Decomposition Metrics:")
print(monthly_data.to_string(index=False))

## 3. Baseline vs Contraction Analysis

Compare April 2018 (T0) vs May 2018 (T1).

In [ ]:
# Extract baseline and contraction periods
t0 = monthly_data[monthly_data['order_month'] == '2018-04'].iloc[0]
t1 = monthly_data[monthly_data['order_month'] == '2018-05'].iloc[0]

print("="*80)
print("REVENUE ROOT CAUSE ANALYSIS: April 2018 (T0) vs May 2018 (T1)")
print("="*80)

# Extract key metrics
u0, f0, aov0, gmv0 = t0['active_users'], t0['order_frequency'], t0['aov'], t0['total_gmv']
u1, f1, aov1, gmv1 = t1['active_users'], t1['order_frequency'], t1['aov'], t1['total_gmv']

# Calculate changes
delta_gmv = gmv1 - gmv0
pct_gmv_change = (delta_gmv / gmv0) * 100
delta_u = u1 - u0
pct_u_change = (delta_u / u0) * 100
delta_f = f1 - f0
pct_f_change = (delta_f / f0) * 100
delta_aov = aov1 - aov0
pct_aov_change = (delta_aov / aov0) * 100

print(f"\n1. USER VOLUME EFFECT")
print(f"   T0 (Apr): {u0:>12,.0f} active users")
print(f"   T1 (May): {u1:>12,.0f} active users")
print(f"   Change:  {delta_u:>12,.0f} users ({pct_u_change:>6.2f}%)")

print(f"\n2. ORDER FREQUENCY EFFECT")
print(f"   T0 (Apr): {f0:>12.3f} orders/user")
print(f"   T1 (May): {f1:>12.3f} orders/user")
print(f"   Change:  {delta_f:>12.3f} ({pct_f_change:>6.2f}%)")

print(f"\n3. AVERAGE ORDER VALUE (AOV) EFFECT")
print(f"   T0 (Apr): BRL{aov0:>11.2f}")
print(f"   T1 (May): BRL{aov1:>11.2f}")
print(f"   Change:  BRL{delta_aov:>11.2f} ({pct_aov_change:>6.2f}%)")

print(f"\n4. GROSS MERCHANDISE VALUE (GMV) RESULT")
print(f"   T0 (Apr): BRL{gmv0:>11.2f}")
print(f"   T1 (May): BRL{gmv1:>11.2f}")
print(f"   Change:  BRL{delta_gmv:>11.2f} ({pct_gmv_change:>6.2f}%)")
print("="*80)

## 4. Segment-Level Contribution Analysis

Identify which customer segments and product categories drove the decline.

In [ ]:
# Geographic segment analysis
geo_analysis = delivered.merge(df_customers[['customer_id', 'customer_state']], on='customer_id', how='left')

geo_by_period = geo_analysis.groupby(['order_month', 'customer_state']).agg({
    'customer_unique_id': 'nunique',
    'order_id': 'count',
    'order_gmv': 'sum'
}).reset_index()

geo_by_period.columns = ['order_month', 'state', 'users', 'orders', 'gmv']

# Pivot for comparison
geo_apr = geo_by_period[geo_by_period['order_month'] == '2018-04'].copy()
geo_may = geo_by_period[geo_by_period['order_month'] == '2018-05'].copy()

geo_comparison = pd.merge(
    geo_apr[['state', 'gmv']].rename(columns={'gmv': 'gmv_apr'}),
    geo_may[['state', 'gmv']].rename(columns={'gmv': 'gmv_may'}),
    on='state',
    how='outer'
)

geo_comparison['gmv_change'] = geo_comparison['gmv_may'] - geo_comparison['gmv_apr']
geo_comparison['pct_change'] = (geo_comparison['gmv_change'] / geo_comparison['gmv_apr'] * 100).round(2)
geo_comparison = geo_comparison.sort_values('gmv_change')

print("\nGEOGRAPHIC SEGMENT IMPACT: GMV Change by State")
print(geo_comparison.to_string(index=False))

# Top losers
top_losers = geo_comparison.nsmallest(5, 'gmv_change')
print(f"\nTop 5 States with Largest GMV Decline:")
print(top_losers[['state', 'gmv_change', 'pct_change']].to_string(index=False))

## 5. Product Category Impact

Analyze which categories contributed to the revenue decline.

In [ ]:
# Category analysis
cat_items = df_order_items.merge(df_products[['product_id', 'category_name_english']], on='product_id', how='left')
cat_orders = cat_items.merge(delivered[['order_id', 'order_month']], on='order_id', how='inner')

cat_items['item_gmv'] = cat_items['price'] + cat_items['freight_value']
cat_orders['item_gmv'] = cat_orders['price'] + cat_orders['freight_value']

cat_by_period = cat_orders.groupby(['order_month', 'category_name_english']).agg({
    'item_gmv': 'sum',
    'order_item_id': 'count'
}).reset_index()

cat_by_period.columns = ['order_month', 'category', 'gmv', 'items']

# Pivot
cat_apr = cat_by_period[cat_by_period['order_month'] == '2018-04'].copy()
cat_may = cat_by_period[cat_by_period['order_month'] == '2018-05'].copy()

cat_comparison = pd.merge(
    cat_apr[['category', 'gmv']].rename(columns={'gmv': 'gmv_apr'}),
    cat_may[['category', 'gmv']].rename(columns={'gmv': 'gmv_may'}),
    on='category',
    how='outer'
)

cat_comparison['gmv_change'] = cat_comparison['gmv_may'] - cat_comparison['gmv_apr']
cat_comparison['pct_change'] = (cat_comparison['gmv_change'] / cat_comparison['gmv_apr'] * 100).round(2)
cat_comparison = cat_comparison.sort_values('gmv_change')

print("\nPRODUCT CATEGORY IMPACT: GMV Change by Category")
print(cat_comparison.to_string(index=False))

# Top losers
top_cat_losers = cat_comparison.nsmallest(5, 'gmv_change')
print(f"\nTop 5 Categories with Largest GMV Decline:")
print(top_cat_losers[['category', 'gmv_change', 'pct_change']].to_string(index=False))

## 6. Waterfall Visualization: Revenue Decomposition

Visualize the three-factor model decomposition.

In [ ]:
# Calculate individual contributions to GMV change
# Using the formula: ΔGMVeff = (Δu * f0 * aov0) + (u1 * Δf * aov0) + (u1 * f1 * Δaov)

user_contribution = delta_u * f0 * aov0
freq_contribution = u1 * delta_f * aov0
aov_contribution = u1 * f1 * delta_aov

# Create waterfall data
categories_waterfall = ['Baseline\nGMV', 'User\nVolume', 'Order\nFrequency', 'AOV\nChange', 'Actual\nGMV']
values = [gmv0, user_contribution, freq_contribution, aov_contribution, gmv1]
colors_waterfall = ['#2E86AB', '#E63946', '#E63946', '#E63946', '#A23B72']

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

cumulative = 0
x_pos = 0

for i, (cat, val) in enumerate(zip(categories_waterfall, values)):
    if i == 0:
        ax.bar(x_pos, val, color=colors_waterfall[i], alpha=0.8, edgecolor='black', linewidth=2)
        cumulative = val
    elif i == len(categories_waterfall) - 1:
        ax.bar(x_pos, val, color=colors_waterfall[i], alpha=0.8, edgecolor='black', linewidth=2)
    else:
        if val < 0:
            ax.bar(x_pos, val, bottom=cumulative, color=colors_waterfall[i], alpha=0.8, edgecolor='black', linewidth=2)
        else:
            ax.bar(x_pos, val, bottom=cumulative, color=colors_waterfall[i], alpha=0.8, edgecolor='black', linewidth=2)
        cumulative += val
    
    # Add value labels
    if i == 0 or i == len(categories_waterfall) - 1:
        ax.text(x_pos, val/2, f'BRL{val:,.0f}', ha='center', va='center', fontweight='bold', fontsize=10, color='white')
    else:
        mid_y = cumulative - val/2
        ax.text(x_pos, mid_y, f'BRL{val:,.0f}\n({val/gmv0*100:.1f}%)', ha='center', va='center', fontweight='bold', fontsize=9)
    
    x_pos += 1

ax.set_xticks(range(len(categories_waterfall)))
ax.set_xticklabels(categories_waterfall)
ax.set_ylabel('GMV (BRL)', fontsize=11, fontweight='bold')
ax.set_title('Revenue Decomposition Waterfall: April → May 2018\nGMV = Users × Frequency × AOV', 
             fontsize=12, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=gmv0, color='gray', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.savefig(BASE_DIR / 'reports' / '06_rca_decomposition_waterfall.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Figure saved to reports/06_rca_decomposition_waterfall.png")

## 7. RCA Summary & Root Causes

Key findings and actionable insights.

In [ ]:
print("\n" + "="*80)
print("ROOT CAUSE ANALYSIS SUMMARY")
print("="*80)

print(f"\nOVERALL IMPACT:")
print(f"  Gross Merchandise Value declined by BRL{abs(delta_gmv):,.2f} ({pct_gmv_change:.2f}%)")
print(f"  From BRL{gmv0:,.2f} (Apr) to BRL{gmv1:,.2f} (May)")

print(f"\nDRIVERS OF DECLINE (Contribution to GMV Change):")
print(f"  1. User Volume Effect:   BRL{user_contribution:>12,.2f} ({user_contribution/delta_gmv*100:>6.1f}% of change)")
print(f"  2. Order Frequency Effect: BRL{freq_contribution:>12,.2f} ({freq_contribution/delta_gmv*100:>6.1f}% of change)")
print(f"  3. AOV Change Effect:    BRL{aov_contribution:>12,.2f} ({aov_contribution/delta_gmv*100:>6.1f}% of change)")

# Determine primary driver
effects = {
    'User Volume': (user_contribution, pct_u_change),
    'Order Frequency': (freq_contribution, pct_f_change),
    'AOV': (aov_contribution, pct_aov_change)
}

primary_driver = min(effects.items(), key=lambda x: x[1][0])

print(f"\nPRIMARY ROOT CAUSE: {primary_driver[0]}")
print(f"  • Change: {primary_driver[1][1]:.2f}%")
print(f"  • Impact on GMV: BRL{primary_driver[1][0]:,.2f}")

print(f"\nTOP IMPACTED SEGMENTS:")
print(f"  Geographic: {top_losers.iloc[0]['state']} (BRL{top_losers.iloc[0]['gmv_change']:,.2f}, {top_losers.iloc[0]['pct_change']:.1f}%)")
print(f"  Category:   {top_cat_losers.iloc[0]['category']} (BRL{top_cat_losers.iloc[0]['gmv_change']:,.2f}, {top_cat_losers.iloc[0]['pct_change']:.1f}%)")

print(f"\n" + "="*80)
print("RECOMMENDATIONS:")
print("="*80)
print("  1. Investigate AOV decline (~4%): Check for pricing errors or promotions")
print("  2. Analyze geographic soft spots: Targeted campaigns for weak states")
print("  3. Review category performance: Strengthen high-margin categories")
print("  4. Monitor order frequency: Loyalty programs to boost repeat purchases")